# 202315163 김소희

## 2026년 10월 1일 지하철 역별 승하차 인원

- 경부선과 경인선을 포함한 수도권 1호선 자료 50건을 출력한다.
- 출력한 50건의 승차 인원과 하차 인원을 각각 더하고, 두 합계도 더한다.


### 조회할 노선과 출력할 자료

서울시 API에서 2026년 10월 1일 `1호선`으로 분류된 자료는 10건이다. 50건을 출력하기 위해 수도권 전철 1호선으로 운행하는 `경부선`과 `경인선` 자료도 함께 조회한다. `1호선 → 경부선 → 경인선` 순서로 자료를 합친 뒤 **앞의 50건**을 출력한다. 각 노선에서는 API가 반환한 순서를 그대로 따른다. 이름에 1호선이 들어가는 `공항철도 1호선`은 제외한다.

각 행에는 같은 날짜에 같은 노선의 한 역에서 승차한 인원과 하차한 인원이 담겨 있다. 서울역처럼 역 이름이 같아도 API에서 서로 다른 노선으로 구분한 자료는 각각 계산한다. **합계는 아래에 출력한 50건만 대상으로 계산한다.**

자료 출처: [서울시 지하철호선별 역별 승하차 인원 정보](https://data.seoul.go.kr/dataList/OA-12914/S/1/datasetView.do?tab=A). 사용한 API는 `CardSubwayStatsNew`이다. 경부선과 경인선이 1호선에 포함되는지는 [코레일 안내](https://info.korail.com/info/selectBbsNttView.do?bbsNo=199&key=911&nttNo=4455)를 참고했다.

### 프로그램

Python 3와 requests를 사용한다. 인증키는 현재 작업 디렉토리 하위의 src/key.properties 파일에 저장된 본인의 dataseoul 값만 사용한다.

In [1]:
from pathlib import Path
from urllib.parse import quote

import requests

USE_YMD = "20261001"
SERVICE = "CardSubwayStatsNew"
LINE_NAMES = ("1호선", "경부선", "경인선")
COUNT = 50


def get_key():
    """현재 작업 디렉토리의 src/key.properties에서 본인의 인증키를 읽는다."""
    key_path = Path("src/key.properties")

    if key_path.exists():
        for line in key_path.read_text(encoding="utf-8-sig").splitlines():
            name, separator, value = line.partition("=")
            if separator and name.strip() == "dataseoul" and value.strip():
                return value.strip()

    raise RuntimeError(
        "현재 작업 디렉토리의 src/key.properties 파일에 "
        "본인의 dataseoul 인증키를 설정하세요."
    )


def get_rows(key, line_name):
    """지정한 날짜와 노선의 역별 승하차 자료를 최대 50건 조회한다."""
    url = "/".join([
        "http://openapi.seoul.go.kr:8088",
        key,
        "json",
        SERVICE,
        "1",
        str(COUNT),
        USE_YMD,
        quote(line_name),
    ])

    try:
        response = requests.get(url, timeout=30)
    except requests.RequestException:
        raise RuntimeError(
            f"{line_name}: 서울시 API 연결에 실패했습니다."
        ) from None

    if response.status_code != 200:
        raise RuntimeError(
            f"{line_name}: HTTP 오류 {response.status_code}"
        )

    try:
        data = response.json()
    except ValueError:
        raise RuntimeError(
            f"{line_name}: JSON 형식의 응답이 아닙니다."
        ) from None

    service_data = data.get(SERVICE, {})
    result = service_data.get("RESULT", data.get("RESULT", {}))

    if result.get("CODE") == "INFO-200":
        return []

    if result.get("CODE") != "INFO-000":
        raise RuntimeError(
            f"{line_name}: API 오류 {result.get('CODE', '응답 코드 없음')}"
        )

    rows = service_data.get("row", [])

    if any(row["USE_YMD"] != USE_YMD for row in rows):
        raise RuntimeError("조회 날짜가 요청 조건과 일치하지 않습니다.")

    # '공항철도 1호선'을 제외하고, 요청한 노선과 정확히 일치하는 자료만 남긴다.
    return [
        row for row in rows
        if row["SBWY_ROUT_LN_NM"] == line_name
    ]


def main():
    key = get_key()
    rows = []

    # 1호선, 경부선, 경인선 순서로 합친다.
    # 각 노선의 자료는 API 응답 순서를 유지한다.
    for line_name in LINE_NAMES:
        rows.extend(get_rows(key, line_name))

    if len(rows) < COUNT:
        raise RuntimeError(
            f"조회된 자료는 {len(rows)}건으로, 필요한 {COUNT}건보다 적습니다."
        )

    selected = rows[:COUNT]
    identities = {
        (
            row["USE_YMD"],
            row["SBWY_ROUT_LN_NM"],
            row["SBWY_STNS_NM"],
        )
        for row in selected
    }

    if len(identities) != COUNT:
        raise RuntimeError(
            "날짜, 노선, 역이 모두 같은 자료가 중복되어 있습니다."
        )

    total_on = 0
    total_off = 0

    print("2026년 10월 1일 수도권 1호선 역별 승하차 인원 (경부선과 경인선 포함)")
    print("출력 순서: 1호선 → 경부선 → 경인선, 앞 50건")
    print("사용일자: 2026-10-01 / 인원 단위: 명")
    print()

    # 출력하는 50건의 승차·하차 인원을 각각 합산한다.
    for number, row in enumerate(selected, start=1):
        on = int(row["GTON_TNOPE"])
        off = int(row["GTOFF_TNOPE"])

        total_on += on
        total_off += off

        print(
            f"{number:02d}. {row['SBWY_ROUT_LN_NM']} | "
            f"{row['SBWY_STNS_NM']} | "
            f"승차 {on:,}명 | 하차 {off:,}명"
        )

    print()
    print(f"출력 건수: {len(selected)}건")

    for line_name in LINE_NAMES:
        count = sum(
            row["SBWY_ROUT_LN_NM"] == line_name
            for row in selected
        )
        print(f"  {line_name}: {count}건")

    print(f"승차 인원 합계: {total_on:,}명")
    print(f"하차 인원 합계: {total_off:,}명")
    print(f"승하차 인원 총합계: {total_on + total_off:,}명")

    return selected, total_on, total_off

### 일괄 실행 결과

아래 셀에서 `main()`을 실행하면 자료를 조회하고, 50건과 합계를 출력한다.


In [2]:
selected_rows, total_on, total_off = main()

2026년 10월 1일 수도권 1호선 역별 승하차 인원 (경부선과 경인선 포함)
출력 순서: 1호선 → 경부선 → 경인선, 앞 50건
사용일자: 2026-10-01 / 인원 단위: 명

01. 1호선 | 서울역 | 승차 88,198명 | 하차 85,260명
02. 1호선 | 시청 | 승차 33,186명 | 하차 33,528명
03. 1호선 | 종각 | 승차 48,251명 | 하차 47,080명
04. 1호선 | 종로3가 | 승차 29,060명 | 하차 25,617명
05. 1호선 | 종로5가 | 승차 25,988명 | 하차 25,574명
06. 1호선 | 동대문 | 승차 13,093명 | 하차 12,627명
07. 1호선 | 신설동 | 승차 16,183명 | 하차 15,520명
08. 1호선 | 제기동 | 승차 17,006명 | 하차 17,397명
09. 1호선 | 청량리(서울시립대입구) | 승차 25,293명 | 하차 25,128명
10. 1호선 | 동묘앞 | 승차 9,914명 | 하차 10,304명
11. 경부선 | 서울역 | 승차 10,269명 | 하차 3,525명
12. 경부선 | 남영 | 승차 11,138명 | 하차 11,707명
13. 경부선 | 용산 | 승차 47,366명 | 하차 47,455명
14. 경부선 | 노량진 | 승차 10,116명 | 하차 10,800명
15. 경부선 | 대방 | 승차 11,758명 | 하차 13,239명
16. 경부선 | 영등포 | 승차 43,054명 | 하차 43,510명
17. 경부선 | 신도림 | 승차 4,384명 | 하차 4,775명
18. 경부선 | 신길 | 승차 9,886명 | 하차 8,891명
19. 경부선 | 구로 | 승차 18,766명 | 하차 18,792명
20. 경부선 | 가산디지털단지 | 승차 20,384명 | 하차 23,303명
21. 경부선 | 금천구청 | 승차 12,093명 | 하차 12,585명
22. 경부선 | 석수 | 승차 9,224명 | 하차 8,495명
23. 경부선 | 관악 | 승